In [38]:
import json
import os
from functools import lru_cache
from typing import Dict, Iterable, List

from time import perf_counter
import torch
import torch.nn.functional as F
from adapters import AutoAdapterModel
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None



In [39]:
DATA_DIR = 'data'
JSONL_PATH = os.path.join(DATA_DIR, 'arxiv_cs_only.jsonl')
INDEX_NAME = 'faiss'
METADATA_PATH = os.path.join(DATA_DIR, 'metadata.json')
EMBED_MODEL_NAME = 'allenai/specter2_base'
RERANK_MODEL_NAME = 'mixedbread-ai/mxbai-rerank-large-v2'


## Document Retrieval

In [40]:
def load_jsonl_papers(path: str) -> List[Dict]:
    papers: List[Dict] = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            abstract = record.get('abstract')
            if abstract is not None:
                record['abstract'] = abstract.strip()
            papers.append(record)
    if not papers:
        raise ValueError(f'No papers found in {path}')
    return papers


In [41]:
if not os.path.exists(JSONL_PATH):
    raise FileNotFoundError(f'Expected corpus at {JSONL_PATH}')

papers = load_jsonl_papers(JSONL_PATH)
print(f'Loaded {len(papers)} papers from {JSONL_PATH}.')
len(papers)


Loaded 110284 papers from data/arxiv_cs_only.jsonl.


110284

## Text Preprocessing & Embedding

In [42]:
class Specter2Embeddings:
    def __init__(self, model_name: str, device: str | None = None, batch_size: int = 16):
        if device is None:
            if torch.cuda.is_available():
                device = 'cuda'
            elif torch.backends.mps.is_available():
                device = 'mps'
            else:
                device = 'cpu'
        self.device = device
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoAdapterModel.from_pretrained(model_name)
        self.model.load_adapter('allenai/specter2', source='hf', load_as='proximity', set_active=True)
        self.model.to(self.device)
        self.model.eval()

    def _prepare_inputs(self, texts: List[str]):
        return self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            return_tensors='pt',
            return_token_type_ids=False,
            max_length=512,
        )

    def _embed_batch(self, inputs):
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        outputs = self.model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
        normalized = F.normalize(cls_embeddings, p=2, dim=1)
        return normalized

    def _embed_texts(self, texts: List[str]) -> List[List[float]]:
        if not texts:
            return []
        embeddings: List[List[float]] = []
        total = len(texts)
        use_progress_bar = tqdm is not None and total > self.batch_size
        progress_bar = tqdm(total=total, desc='Embedding texts', unit='doc') if use_progress_bar else None
        log_progress = tqdm is None and total > self.batch_size
        log_step = max(self.batch_size, total // 10 or 1) if log_progress else None
        next_log = log_step if log_progress else None
        if log_progress:
            print(f'Embedding {total} texts...')
        processed = 0
        try:
            with torch.inference_mode():
                for start_idx in range(0, total, self.batch_size):
                    batch = texts[start_idx:start_idx + self.batch_size]
                    inputs = self._prepare_inputs(batch)
                    normalized = self._embed_batch(inputs)
                    embeddings.extend(normalized.cpu().tolist())
                    batch_size = len(batch)
                    processed += batch_size
                    if progress_bar is not None:
                        progress_bar.update(batch_size)
                    elif log_progress and processed >= next_log:
                        print(f'Embedded {processed}/{total} texts')
                        next_log += log_step
        finally:
            if progress_bar is not None:
                progress_bar.close()
        if log_progress:
            if processed < total:
                print(f'Embedded {processed}/{total} texts')
            print('Embedding complete.')
        return embeddings

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self._embed_texts(texts)

    def embed_query(self, text: str) -> List[float]:
        return self._embed_texts([text])[0]


    def __call__(self, texts):
        if isinstance(texts, str):
            return self.embed_query(texts)
        try:
            iterable = list(texts)
        except TypeError as exc:
            raise TypeError('Expected iterable of strings or a single string.') from exc
        return self.embed_documents(iterable)

@lru_cache(maxsize=1)
def get_embedder(model_name: str = EMBED_MODEL_NAME) -> Specter2Embeddings:
    return Specter2Embeddings(model_name=model_name)


def prepare_corpus_texts(papers: List[Dict], sep_token: str) -> List[str]:
    prepared: List[str] = []
    for paper in papers:
        title = (paper.get('title') or '').strip()
        abstract = (paper.get('abstract') or '').strip()
        joined = f"{title}{sep_token}{abstract}" if title or abstract else ''
        prepared.append(joined)
    return prepared


## Database Creation

In [43]:
def build_faiss_index(papers: List[Dict]) -> None:
    os.makedirs(DATA_DIR, exist_ok=True)
    embedder = get_embedder()
    sep = embedder.tokenizer.sep_token or ' '
    texts = prepare_corpus_texts(papers, sep_token=sep)
    metadatas = []
    for paper, text in zip(papers, texts):
        authors = paper.get('authors')
        if not authors and paper.get('authors_parsed'):
            authors = ', '.join(' '.join(filter(None, author)).strip() for author in paper['authors_parsed'])
        metadatas.append(
            {
                'id': paper.get('id'),
                'title': paper.get('title'),
                'abstract': paper.get('abstract'),
                'authors': authors,
                'categories': paper.get('categories'),
                'update_date': paper.get('update_date'),
                'content': text,
            }
        )

    vectorstore = FAISS.from_texts(texts=texts, embedding=embedder, metadatas=metadatas)
    vectorstore.save_local(DATA_DIR, index_name=INDEX_NAME)

    with open(METADATA_PATH, 'w', encoding='utf-8') as f:
        json.dump(metadatas, f, ensure_ascii=False, indent=2)

    print('Index and metadata saved to', DATA_DIR)


In [44]:
# build_faiss_index(papers)

## FAISS Local Loading

In [45]:
def load_vectorstore() -> FAISS:
    index_path = os.path.join(DATA_DIR, f"{INDEX_NAME}.faiss")
    store_path = os.path.join(DATA_DIR, f"{INDEX_NAME}.pkl")
    if not (os.path.exists(index_path) and os.path.exists(store_path)):
        raise FileNotFoundError('Run the database creation step first.')

    embedder = get_embedder()
    return FAISS.load_local(
        DATA_DIR,
        embedder,
        index_name=INDEX_NAME,
        allow_dangerous_deserialization=True,
    )

## Test Query

In [46]:
def semantic_search(query: str, top_k: int = 5, fetch_k: int | None = None) -> List[Dict]:
    vectorstore = load_vectorstore()
    fetch = fetch_k or max(top_k, 20)
    fetch = max(fetch, top_k)

    docs_with_scores = vectorstore.similarity_search_with_score(query, k=fetch)

    results = []
    for doc, score in docs_with_scores:
        doc_info = dict(doc.metadata)
        doc_info.setdefault('content', doc.page_content)
        doc_info['vector_score'] = float(score)
        results.append(doc_info)
    return results[:top_k]

In [47]:
query = 'large language models for translation'
results = semantic_search(query, top_k=5)
for rank, item in enumerate(results, start=1):
    print(f"{rank}. {item['title']} (vector={item['vector_score']:.3f})")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

There are adapters available but none are activated for the forward pass.
`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


1. Learning Translation Quality Evaluation on Low Resource Languages from
  Large Language Models (vector=0.074)
2. Survey of Low-Resource Machine Translation (vector=0.077)
3. A Paradigm Shift in Machine Translation: Boosting Translation
  Performance of Large Language Models (vector=0.077)
4. Beyond Decoder-only: Large Language Models Can be Good Encoders for Machine Translation (vector=0.078)
5. Many-to-English Machine Translation Tools, Data, and Pretrained Models (vector=0.080)


## Reranker

In [48]:
@lru_cache(maxsize=1)
def get_reranker(model_name: str = RERANK_MODEL_NAME) -> CrossEncoder:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    return CrossEncoder(model_name, device=device)


def rerank_results(query: str, results: List[Dict], top_k: int | None = None) -> List[Dict]:
    if not results:
        return []

    reranker = get_reranker()
    passages = [
        doc.get('content')
        or doc.get('abstract')
        or f"Title: {doc.get('title', '')}"
        for doc in results
    ]
    scores = reranker.predict([[query, passage] for passage in passages])

    enriched = []
    for doc, score in zip(results, scores):
        doc_copy = dict(doc)
        doc_copy['rerank_score'] = float(score)
        enriched.append(doc_copy)

    enriched.sort(key=lambda x: x['rerank_score'], reverse=True)
    if top_k is not None:
        enriched = enriched[:top_k]
    return enriched


In [49]:
query = 'graph neural networks for NLP'
print('Query:', query)

vector_hits = semantic_search(query, top_k=5, fetch_k=25)
reranked_hits = rerank_results(query, vector_hits, top_k=5)

print('-- MXBAI reranked --')
for rank, item in enumerate(reranked_hits, start=1):
    print(f"{rank}. {item['title']} (rerank={item['rerank_score']:.3f} | vector={item['vector_score']:.3f})")

print('-- Vector-only --')
for rank, item in enumerate(vector_hits, start=1):
    print(f"{rank}. {item['title']} (vector={item['vector_score']:.3f})")

Query: graph neural networks for NLP


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.
Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at mixedbread-ai/mxbai-rerank-large-v2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


-- MXBAI reranked --
1. A Decade of Knowledge Graphs in Natural Language Processing: A Survey (rerank=0.983 | vector=0.095)
2. Neural Graph Embedding Methods for Natural Language Processing (rerank=0.966 | vector=0.096)
3. Tackling Graphical NLP problems with Graph Recurrent Networks (rerank=0.956 | vector=0.090)
4. Graph Neural Networks for Natural Language Processing: A Survey (rerank=0.919 | vector=0.050)
5. Graph Language Models (rerank=0.919 | vector=0.089)
-- Vector-only --
1. Graph Neural Networks for Natural Language Processing: A Survey (vector=0.050)
2. Graph Language Models (vector=0.089)
3. Tackling Graphical NLP problems with Graph Recurrent Networks (vector=0.090)
4. A Decade of Knowledge Graphs in Natural Language Processing: A Survey (vector=0.095)
5. Neural Graph Embedding Methods for Natural Language Processing (vector=0.096)


In [52]:
while True:
    try:
        query = input('Enter query (press Enter to exit): ').strip()
    except EOFError:
        print('Stopping interactive search.')
        break
    if not query:
        print('Stopping interactive search.')
        break

    k_input = input('Enter k (number of results to display): ').strip()
    if not k_input:
        k = 5
    else:
        try:
            k = int(k_input)
        except ValueError:
            print('Please provide an integer for k.')
            continue
    if k <= 0:
        print('k should be a positive integer.')
        continue

    print("Query: ", query + "\n" + str(k))

    fetch_k = max(k * 5, k)
    retrieve_start = perf_counter()
    vector_hits = semantic_search(query, top_k=fetch_k, fetch_k=fetch_k)
    retrieve_elapsed = perf_counter() - retrieve_start

    rerank_start = perf_counter()
    reranked_hits = rerank_results(query, vector_hits, top_k=k)
    rerank_elapsed = perf_counter() - rerank_start
    total_elapsed = retrieve_elapsed + rerank_elapsed
    print(f'Retrieval {retrieve_elapsed:.3f}s | Rerank {rerank_elapsed:.3f}s | Total {total_elapsed:.3f}s')

    if not reranked_hits:
        print('No results found.')
        continue

    print(f'-- Top {len(reranked_hits)} reranked results --')
    for rank, item in enumerate(reranked_hits, start=1):
        title = item.get('title') or 'Untitled'
        rerank_score = item.get('rerank_score')
        vector_score = item.get('vector_score')
        rerank_text = f'{rerank_score:.3f}' if isinstance(rerank_score, (int, float)) else 'n/a'
        vector_text = f'{vector_score:.3f}' if isinstance(vector_score, (int, float)) else 'n/a'
        abstract = (item.get('abstract') or '').strip()
        print(f"{rank}. {title} (rerank={rerank_text} | vector={vector_text})")
        if abstract:
            preview = abstract.replace('\n', ' ')
            print(f'   Abstract: {preview}')


Query:  How can AI help researchers fine related scientific literature
10


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


Retrieval 1.155s | Rerank 19.415s | Total 20.570s
-- Top 10 reranked results --
1. Related Work and Citation Text Generation: A Survey (rerank=0.990 | vector=0.143)
   Abstract: To convince readers of the novelty of their research paper, authors must perform a literature review and compose a coherent story that connects and relates prior works to the current work. This challenging nature of literature review writing makes automatic related work generation (RWG) academically and computationally interesting, and also makes it an excellent test bed for examining the capability of SOTA natural language processing (NLP) models. Since the initial proposal of the RWG task, its popularity has waxed and waned, following the capabilities of mainstream NLP approaches. In this work, we survey the zoo of RWG historical works, summarizing the key approaches and task definitions and discussing the ongoing challenges of RWG.
2. AppTechMiner: Mining Applications and Techniques from Scientific
  Article